In [1]:
COCO_JSON_PATH = "/content/drive/MyDrive/Tray Track Dataset/instances_COCO.json"
IMAGES_DIR = "/content/drive/MyDrive/Tray Track Dataset/Full Menu Images"
OUTPUT_DIR = "/content/drive/MyDrive/Tray Track Dataset/Augmented Single Item Images"
OUTPUT_JSON = "/content/drive/MyDrive/Tray Track Dataset/newMasks.json"

In [2]:
import os
import json
import cv2
import numpy as np
from pycocotools import mask as maskUtils
from typing import Any

### I / O Helpers

In [3]:
def load_coco(json_path: str) -> dict:
    with open(json_path) as f:
        return json.load(f)

In [4]:
def save_coco(coco: dict, out_path: str) -> None:
    with open(out_path, "w") as f:
        json.dump(coco, f, indent=2)

In [5]:
def build_index(coco: dict) -> tuple[dict, dict]:
    images_dict     = {img["id"]: img for img in coco["images"]}
    categories_dict = {cat["id"]: cat["name"] for cat in coco["categories"]}
    return images_dict, categories_dict

### white background size

In [6]:
def compute_canvas_size(coco: dict) -> tuple[int, int]:
    """Return (width, height) of the largest image in the dataset."""
    max_w = max(img["width"]  for img in coco["images"])
    max_h = max(img["height"] for img in coco["images"])
    return max_w, max_h

### Decoding

In [7]:
def _decode_rle(rle: dict, h: int, w: int) -> np.ndarray:
    if isinstance(rle["counts"], list):
        rle = maskUtils.frPyObjects(rle, h, w)
    return maskUtils.decode(rle).astype(np.uint8) * 255

In [8]:
def _draw_segment(mask: np.ndarray, seg: list | dict,
                  h: int, w: int, shape_type: str, ann_id: Any) -> None:
    if isinstance(seg, dict):
        np.maximum(mask, _decode_rle(seg, h, w), out=mask)
        return

    flat = np.array(seg, dtype=np.float32)
    pts  = flat.reshape(-1, 2)

    if len(flat) == 5 or shape_type == "ellipse":
        try:
            cx, cy, rx, ry, angle = flat[:5]
            cv2.ellipse(mask, (int(cx), int(cy)),
                        (int(abs(rx)), int(abs(ry))),
                        float(angle), 0, 360, 255, -1)
        except Exception as e:
            print(f"⚠️  Ellipse draw failed (ann {ann_id}): {e}")

    elif len(pts) == 2 or (len(flat) == 4 and shape_type in ("rectangle", "points")):
        x1 = int(min(pts[0][0], pts[1][0]))
        y1 = int(min(pts[0][1], pts[1][1]))
        x2 = int(max(pts[0][0], pts[1][0]))
        y2 = int(max(pts[0][1], pts[1][1]))
        cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)

    elif len(pts) >= 3:
        cv2.fillPoly(mask, [pts.astype(np.int32)], 255)

    else:
        print(f"⚠️  Unrecognised segment (ann {ann_id}): {len(flat)} values")


In [9]:
def decode_mask(ann: dict, h: int, w: int) -> np.ndarray:
    mask       = np.zeros((h, w), dtype=np.uint8)
    seg        = ann.get("segmentation")
    shape_type = ann.get("attributes", {}).get("type", "polygon")

    if isinstance(seg, list) and seg:
        for s in seg:
            _draw_segment(mask, s, h, w, shape_type, ann["id"])
    elif isinstance(seg, dict):
        mask = _decode_rle(seg, h, w)
    else:
        print(f"ℹ️  No segmentation for ann {ann['id']}, falling back to bbox")
        x, y, bw, bh = map(int, ann["bbox"])
        cv2.rectangle(mask, (x, y), (x + bw, y + bh), 255, -1)

    return mask

### Rotation

In [10]:
def rotate_image_and_mask(img: np.ndarray, mask: np.ndarray,
                           angle_deg: float) -> tuple[np.ndarray, np.ndarray]:
    """
    Rotate both image and mask by angle_deg around centre.
    Canvas expands to fit full rotated content — no cropping.
    """
    h, w   = img.shape[:2]
    cx, cy = w / 2.0, h / 2.0
    M      = cv2.getRotationMatrix2D((cx, cy), angle_deg, 1.0)

    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)

    M[0, 2] += (new_w / 2.0) - cx
    M[1, 2] += (new_h / 2.0) - cy

    rotated_img  = cv2.warpAffine(img,  M, (new_w, new_h),
                                  flags=cv2.INTER_LINEAR,
                                  borderValue=(255, 255, 255))
    rotated_mask = cv2.warpAffine(mask, M, (new_w, new_h),
                                  flags=cv2.INTER_NEAREST,
                                  borderValue=0)
    return rotated_img, rotated_mask

### Place on Background

In [11]:
def place_on_canvas(img: np.ndarray, mask: np.ndarray,
                    canvas_w: int, canvas_h: int
                    ) -> tuple[np.ndarray, np.ndarray, int, int]:
    """
    Centre img and mask on a white canvas of (canvas_w x canvas_h).
    If the rotated content is larger than the canvas, scale it down
    to fit (preserving aspect ratio) before placing.
    Returns (canvas_img, canvas_mask, offset_x, offset_y).
    """
    h, w = img.shape[:2]

    # Scale down if content exceeds canvas in either dimension
    scale = min(canvas_w / w, canvas_h / h, 1.0)  # 1.0 = never upscale
    if scale < 1.0:
        new_w = int(w * scale)
        new_h = int(h * scale)
        img   = cv2.resize(img,  (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        mask  = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
        h, w  = new_h, new_w

    off_x = (canvas_w - w) // 2
    off_y = (canvas_h - h) // 2

    canvas_img  = np.full((canvas_h, canvas_w, 3), 255, dtype=np.uint8)
    canvas_mask = np.zeros((canvas_h, canvas_w),   dtype=np.uint8)

    canvas_img [off_y:off_y + h, off_x:off_x + w] = img
    canvas_mask[off_y:off_y + h, off_x:off_x + w] = mask

    return canvas_img, canvas_mask, off_x, off_y

### masks to COCO file builder

In [12]:
def mask_to_coco_segmentation(mask: np.ndarray) -> tuple[dict, list, float]:
    """
    Convert a uint8 mask → (rle, bbox, area) ready for a COCO annotation.
    bbox is [x, y, w, h].
    """
    binary = (mask > 0).astype(np.uint8)
    rle    = maskUtils.encode(np.asfortranarray(binary))
    rle["counts"] = rle["counts"].decode("utf-8")

    coords = cv2.findNonZero(mask)
    if coords is not None:
        x, y, bw, bh = cv2.boundingRect(coords)
        bbox = [float(x), float(y), float(bw), float(bh)]
    else:
        bbox = [0.0, 0.0, 0.0, 0.0]

    area = float(maskUtils.area(maskUtils.encode(np.asfortranarray(binary))))
    return rle, bbox, area

### Pipeline

In [13]:
def extract_rotate_export(
    coco_json_path:    str,
    images_dir:        str,
    output_images_dir: str,
    output_json_path:  str,
    rotation_angles:   list[float],        # e.g. [15, 30, 45, 90, 180]
    category_output_dirs: bool = True,     # save into per-category subfolders
) -> None:
    """
    For every annotation:
      1. Decode mask from COCO segmentation.
      2. Apply mask → object pixels on white background (source image size).
      3. Rotate by each angle; expand canvas to fit rotated content.
      4. Centre result on a white canvas sized to the dataset's largest image.
      5. Save PNG and record a new COCO annotation (RLE segmentation).
    """
    os.makedirs(output_images_dir, exist_ok=True)

    coco                     = load_coco(coco_json_path)
    images_dict, categories_dict = build_index(coco)
    canvas_w, canvas_h       = compute_canvas_size(coco)

    print(f"Canvas size: {canvas_w} × {canvas_h}  "
          f"({len(coco['annotations'])} annotations × "
          f"{len([0] + rotation_angles)} variants)")

    if category_output_dirs:
        for cat_name in categories_dict.values():
            os.makedirs(os.path.join(output_images_dir, cat_name), exist_ok=True)

    angles      = [0.0] + rotation_angles   # always include the original
    out_images  = []
    out_anns    = []
    new_img_id  = 1
    new_ann_id  = 1

    for ann in coco["annotations"]:
        img_info = images_dict[ann["image_id"]]
        img_path = os.path.join(images_dir, img_info["file_name"])

        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️  Could not read: {img_path}")
            continue

        h, w = img.shape[:2]
        mask = decode_mask(ann, h, w)

        if mask.max() == 0:
            print(f"⚠️  Empty mask for ann {ann['id']} — skipping")
            continue

        # Mask out background → white
        white      = np.full_like(img, 255)
        masked_img = np.where(mask[:, :, None] == 255, img, white)

        cat_name   = categories_dict.get(ann["category_id"], "unknown")
        base_stem  = (f"{os.path.splitext(img_info['file_name'])[0]}"
                      f"__ann{ann['id']}")

        for angle in angles:
            # 1. Rotate (expand canvas to avoid cropping)
            if angle != 0.0:
                rot_img, rot_mask = rotate_image_and_mask(masked_img, mask, angle)
            else:
                rot_img, rot_mask = masked_img, mask

            # 2. Centre on dataset-wide white canvas
            canvas_img, canvas_mask, _, _ = place_on_canvas(
                rot_img, rot_mask, canvas_w, canvas_h)

            # 3. Build COCO annotation from canvas mask
            rle, bbox, area = mask_to_coco_segmentation(canvas_mask)

            # 4. Save image
            angle_tag = f"_rot{int(angle)}" if angle != 0.0 else ""
            fname     = f"{base_stem}{angle_tag}.png"
            fpath     = (os.path.join(output_images_dir, cat_name, fname)
                         if category_output_dirs
                         else os.path.join(output_images_dir, fname))
            cv2.imwrite(fpath, canvas_img)

            # 5. Record entries
            rel_path = os.path.join(cat_name, fname) if category_output_dirs else fname
            out_images.append({
                "id":        new_img_id,
                "file_name": rel_path,
                "width":     canvas_w,
                "height":    canvas_h,
            })
            out_anns.append({
                "id":           new_ann_id,
                "image_id":     new_img_id,
                "category_id":  ann["category_id"],
                "segmentation": rle,
                "bbox":         bbox,
                "area":         area,
                "iscrowd":      0,
            })
            new_img_id += 1
            new_ann_id += 1

    out_coco = {
        "images":      out_images,
        "annotations": out_anns,
        "categories":  coco["categories"],
    }
    save_coco(out_coco, output_json_path)
    print(f"✅ Done — {len(out_images)} images saved → {output_json_path}")


In [14]:
if __name__ == "__main__":
    extract_rotate_export(
        coco_json_path    = COCO_JSON_PATH,
        images_dir        = IMAGES_DIR,
        output_images_dir = OUTPUT_DIR,
        output_json_path  = OUTPUT_JSON,
        rotation_angles   = [15, 30, 45, 60, 75, 90, 105, 120, 135, 150,
                             165, 180, 195, 210, 225, 240, 255, 270, 285,
                             300, 315, 330, 345],
        category_output_dirs = True,
    )

Canvas size: 1179 × 2096  (574 annotations × 24 variants)
ℹ️  No segmentation for ann 4, falling back to bbox
ℹ️  No segmentation for ann 9, falling back to bbox
ℹ️  No segmentation for ann 19, falling back to bbox
ℹ️  No segmentation for ann 21, falling back to bbox
ℹ️  No segmentation for ann 24, falling back to bbox
ℹ️  No segmentation for ann 25, falling back to bbox
ℹ️  No segmentation for ann 26, falling back to bbox
ℹ️  No segmentation for ann 27, falling back to bbox
ℹ️  No segmentation for ann 28, falling back to bbox
ℹ️  No segmentation for ann 111, falling back to bbox
ℹ️  No segmentation for ann 112, falling back to bbox
ℹ️  No segmentation for ann 113, falling back to bbox
ℹ️  No segmentation for ann 117, falling back to bbox
ℹ️  No segmentation for ann 124, falling back to bbox
ℹ️  No segmentation for ann 142, falling back to bbox
ℹ️  No segmentation for ann 143, falling back to bbox
ℹ️  No segmentation for ann 144, falling back to bbox
ℹ️  No segmentation for ann 146, fa

In [ ]:
# !zip -r output_white_bg.zip output_white_bg/
# from google.colab import files
# files.download(f"{OUTPUT_DIR}.zip")